# **MNIST - CNN for Handwritten Digit Recognition V.1**

# **1 - Libraries**


In [ ]:
import os
import numpy as np 
import pandas as pd
import matplotlib.pyplot as plt
import random

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras.models import Sequential
from tensorflow.keras.utils import to_categorical
from tensorflow.keras.layers import Dense, Conv2D, MaxPool2D, Activation, Dropout, BatchNormalization, Flatten
from tensorflow.keras.preprocessing.image import ImageDataGenerator

from sklearn.model_selection import train_test_split

In [ ]:
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

In [ ]:
print("Num GPUs Available: ", len(tf.config.experimental.list_physical_devices('GPU')))
print("Tensorflow version: ", tf.__version__)

# **2 - DATA**

## 2.1 Loading Data


In [ ]:
# file path
train_file_path = "/kaggle/input/digit-recognizer/train.csv"
test_file_path = "/kaggle/input/digit-recognizer/test.csv"

# loading
train_data = pd.read_csv(train_file_path)
test_data = pd.read_csv(test_file_path)

## 2.2 Data Preprocessing

In [ ]:
y_train_full = train_data["label"]
X_train_full = train_data.drop(labels="label", axis=1)
X_test = test_data 

# normalizing 
X_train_full = X_train_full/225
X_test = X_test/225

# .values : convert a pandas DataFrame into a numpy Array
# .reshape : take 3 first arg as the shape of the np Array, the last arg : 1=grayscale, 3=RGB
X_train_full = X_train_full.values.reshape(-1,28,28,1)
X_test = X_test.values.reshape(-1,28,28,1)

# one_hot encoding
y_train_full = to_categorical(y_train_full, num_classes=10)


In [ ]:
# split the full training data into 2 seperate set for training and validation
X_train,  X_val, y_train, y_val = train_test_split(
    X_train_full, y_train_full, 
    test_size=0.1, 
    random_state=0, 
    stratify=y_train_full)

In [ ]:
print(y_train_full[1])

In [ ]:
print(X_train.shape)
print(y_train.shape)

## 2.3 Data Augmentation

In [ ]:
# Setup Generator
data_gen = ImageDataGenerator(
    rotation_range=10,
    zoom_range=0.1,
    width_shift_range=0.1,
    height_shift_range=0.1   
)

In [ ]:
# Preview Augmented Images
a = 8 # a: number of sample we take for augmentation procedure
b = 8 # b: number of augmentation we implement on a single sample 

fig, axes = plt.subplots(a, b, figsize=(a, b))
fig.tight_layout(pad=0.01)

random_indices = random.sample(range(X_train.shape[0]), a)

for i in range(a):
    for j in range(b):
        sample_X = X_train[random_indices[i]].reshape((1,28,28,1))
        sample_y = y_train[random_indices[i]].reshape((1,10))
        
        aug_X, aug_y = next(data_gen.flow(sample_X, sample_y, batch_size=1))

        ax = axes[i, j]
        ax.imshow(aug_X.reshape(28,28), cmap='gray')
        
        ax.set_axis_off()


## 2.4 Data Visualization

In [ ]:
# def create_image_displayer(row, col, )

In [ ]:
fig, axes = plt.subplots(8, 8, figsize=(8,8))
fig.tight_layout(pad=0.01)

m = X_train.shape[0]

for i, ax in enumerate(axes.flat):
    random_id = np.random.randint(m)
    # reshape the vector into an image
    reshaped_random_X_train = X_train[random_id].reshape((28,28))

    ax.imshow(reshaped_random_X_train, cmap='gray')
    ax.set_title(int(np.argmax(y_train[random_id])))
    ax.set_axis_off()

# **3 - CNN Model**

## 3.1 Define the Model

In [ ]:
# set up some parameters
batch_size = 64
epochs = 500
input_shape = X_train.shape[1:]

print(input_shape)

In [ ]:
model_1 = Sequential([
    Conv2D(filters=32, kernel_size=3, input_shape=input_shape, padding='same'),
    #BatchNormalization(),
    Activation("relu"),
    MaxPool2D(pool_size=(2,2), strides=2, padding='same'),

    Conv2D(filters=64, kernel_size=3, padding='same'),
    #BatchNormalization(),
    Activation("relu"),
    MaxPool2D(pool_size=(2,2), strides=2, padding='same'),

    Conv2D(filters=128, kernel_size=3, padding='same'),
    #BatchNormalization(),
    Activation("relu"),
    MaxPool2D(pool_size=(2,2), strides=2, padding='same'),

    Flatten(),

    Dense(units=128, activation='relu'),
    #Dropout(0.3),
    Dense(units=10, activation='softmax')  
])

In [ ]:
model_2 = Sequential([
    Conv2D(filters=32, kernel_size=5, input_shape=input_shape, padding='same', activation='relu'),
    Conv2D(filters=32, kernel_size=5, input_shape=input_shape, padding='same', activation='relu'),
    MaxPool2D(pool_size=(2,2), strides=2, padding='same'),
    Dropout(0.25),

    Conv2D(filters=64, kernel_size=3, input_shape=input_shape, padding='same', activation='relu'),
    Conv2D(filters=64, kernel_size=3, input_shape=input_shape, padding='same', activation='relu'),
    MaxPool2D(pool_size=(2,2), strides=2, padding='same'),
    Dropout(0.25),

    Flatten(),

    Dense(units=128, activation='relu'),
    Dropout(0.5),
    Dense(units=10, activation='softmax')
    
])

In [ ]:
model_3 = Sequential([
    Conv2D(filters=32, kernel_size=3, activation='relu', input_shape=input_shape),
    BatchNormalization(),
    Conv2D(filters=32, kernel_size=3, activation='relu'),
    BatchNormalization(),
    Conv2D(filters=32, kernel_size=5, strides=2, padding='same', activation='relu'),
    BatchNormalization(),
    Dropout(0.4),
    
    Conv2D(filters=64, kernel_size=3, activation='relu'),
    BatchNormalization(),
    Conv2D(filters=64, kernel_size=3, activation='relu'),
    BatchNormalization(),
    Conv2D(filters=64, kernel_size=5, strides=2, padding='same', activation='relu'),
    BatchNormalization(),
    Dropout(0.4),

    Conv2D(filters=128, kernel_size=4, activation='relu'),
    BatchNormalization(),
    
    Flatten(),

    Dropout(0.4),
    Dense(units=10, activation='softmax')
    
])

In [ ]:
model = model_1

In [ ]:
model.summary()

In [ ]:
model.compile(optimizer='adam', loss="categorical_crossentropy", metrics=["accuracy"])

## 3.2 Fit the Training Data

In [ ]:
early_stopping = keras.callbacks.EarlyStopping(
    patience=10,
    min_delta=0.001,
    restore_best_weights=True
)

In [ ]:
history = model.fit(
    data_gen.flow(X_train, y_train),
    # X_train, y_train,
    validation_data=(X_val, y_val),
    steps_per_epoch=len(X_train)//batch_size,
    epochs=epochs,
    callbacks=[early_stopping]
)

# **4 - Evaluation**

## 4.1 Sample Prediction

In [ ]:
i = 1500

preds = model.predict(X_train[i].reshape(-1,28,28,1))
print(int(np.argmax(preds)))
print(int(np.argmax(y_train[i])))
plt.imshow(X_train[i].reshape((28,28)))

# **5 - Submission**

In [ ]:
# making predictions 

y_pred = model.predict(X_test)
y_pred_submission = y_pred.argmax(axis=1)

In [ ]:
# output submission

output_df = pd.DataFrame({
    "ImageId": range(1, len(y_pred_submission)+1),
    "Label": y_pred_submission
})

output_file_path = "/kaggle/working/cnn_for_handwritten_digit_submission.csv"
output_df.to_csv(output_file_path, index=False)

score = model.evaluate(X_val, y_val)
print('Test loss: ', score[0])
print('Test accuracy: ', score[1])